# TO DO

- For sites with changing coordinates (Green River and Pelican) make sure I use the correct coordinates

# Background

Author: Diane Menuz, based on script from Paul Inkenbrandt  
Date: 8/6/2026  
  
Purpose: Template for calculating ET-weighted daily and monthly footprint values using  
ffp_xr, which is the closest model to the published Kljun Python footprint code. Example  
includes calculating z0 and zm, weighting by NLDAS ETo values (taking data from the  
[nldas notebook example](docs\notebooks\nldas_et_download_example.ipynb)), and then calculating and exporting daily and monthly  
footprints. 
  
Notes
- Raster export may take a long time if you export to a networked drive
- Currently code creates footprints for all data in the database, but could be modified  
to only create footprints for a user-specified date range
- This script assumes that the index for the data is datetime_end and creates a new   
datetime_start index based on the DATA_FREQ

## Parameters and Libraries

In [1]:
# paths to footprint library and database connection
footprint_path = "C:/Users/Hutto/OneDrive/Documents/Github/footprints/src"

In [2]:
import gc
import warnings

warnings.filterwarnings("ignore")

import pathlib
import sys
from pathlib import Path

# Add the footprint path to the system path
sys.path.append(footprint_path)

import contextily as cx
import matplotlib.pyplot as plt
import pandas as pd
from fluxfootprints import (
    animate_footprint,
    build_climatology,
    compute_aerodynamic_params,
    export_rasters_geotiff,
    summarize_periods,
)


* US-UTV
* US-UTD
* US-UTE
* US-UTG
* US-UTJ
* US-UTL
* US-UTV
* US-UTW
* US-UTM

In [3]:
# Setting vegetation presets and heights for each station, based on field measurements and literature values

VEG_PARAMS = {'US-UTW': ['alfalfa', 0.5],
              'US-UTM': ['wetland_grass', 1.5],
              'US-UTL': ['alfalfa', 0.5],
              'US-UTB': ['playa', 0.0],
              'US-UTD': ['alfalfa', 0.5],
              'US-UTV': ['alfalfa', 0.5],
              'US-UTG': ['corn_standard', 0.5],
              'US-UTP': ['phragmites', 3.0],
              'US-UTX': ['pinyon_juniper', 2.0],
              'US-UTY': ['open_water', 0.0],
              'US-UTH': ['open_water', 0.0],
              'US-CdM': ['pinyon_juniper', 3.0],
              'US-UTE': ['alfalfa', 0.5],
                       }

In [ ]:
# stations to process; each needs an NLDAS csv and should have an entry in VEG_PARAMS
STATIONS = ['US-UTD', 'US-UTW', 'US-UTE', 'US-UTL']
# subset of data for processing footprints, must include the time!
# if you want all data in the database, make bounds larger than dates in the database
START = '2020-01-01 0:00'
END = '2025-12-31 23:30'
# time of data subset for data, 8 to 20 follows Volk (2023)
START_TIME = 8
END_TIME = 20
DX, DY = 10.0, 10.0   # output raster resolution; John used 30 m
DOMAIN = (-600.0, 600.0, -600.0, 600.0)   # John used 600 x 600
DATA_FREQ = '30min' # set to the data frequency, used to resample nldas et data to match tower data

In [ ]:
# folder locations; station subfolders ({stationid}/weighted, {stationid}/non_weighted) are created per station
db_connection_path = "M:/My Drive/Python/dbconnect.py"
FP_DIR = pathlib.Path("M:/My Drive/footprints/")
NLDAS_DIR = pathlib.Path("M:/Shared drives/UGS_Flux/Data_Processing/final_processing_workflow/footprints/nldas/nldas_results")
DOMAIN_DIR = Path('D:/footprints/domain_outputs/')
DOMAIN_DIR.mkdir(parents=True, exist_ok=True)

## Read in Database Data

In [6]:
import importlib

spec = importlib.util.spec_from_file_location("dbconnect", db_connection_path)
dbconnect = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dbconnect)

engine = dbconnect.postconn_ugs_write()

In [ ]:
def read_station_data(stationid):
    """Query location, half-hourly data, footprint parameters and IRGA height for one station."""
    latlong_query = f"""SELECT stationid, latitude, longitude
                    FROM groundwater.station_attribute_view
                    where stationid = '{stationid}'
                """
    locations = pd.read_sql(latlong_query, engine)
    if locations.empty:
        raise ValueError(f"No location found for {stationid}")
    station_lat = pd.to_numeric(locations["latitude"], errors="coerce").iloc[0]
    station_lon = pd.to_numeric(locations["longitude"], errors="coerce").iloc[0]

    stationdat_query = f"""SELECT *
                    FROM groundwater.metflux_qc
                    where stationid = '{stationid}'
                """
    df_temp = pd.read_sql(stationdat_query, engine)
    df_temp['datetime_start'] = df_temp.datetime_end - pd.Timedelta(DATA_FREQ)
    df_temp = df_temp.reset_index().set_index('datetime_start')

    param_query = f"""SELECT *
                    FROM groundwater.daily_footprint_parameters
                    where stationid = '{stationid}'
                    and zone_status = 'primary'
                """
    params = pd.read_sql(param_query, engine)
    params['height_date'] = pd.to_datetime(params['height_date'])
    params = params.set_index('height_date')
    params = params.resample(DATA_FREQ).ffill()

    irga_query = f"""SELECT variable_value
                    FROM groundwater.dynamic_station_data
                    WHERE variable_name = 'irgason'
                    AND stationid = '{stationid}'"""
    irga = pd.read_sql(irga_query, engine)['variable_value']
    irga_height = irga.iloc[0] if len(irga) else None

    return df_temp, params, irga_height, station_lat, station_lon

## Prep Data for Footprints

- Compute aerodynamic parameters (z0 and zm)
- Read in NLDAS ET values and join to eddy covariance data frame
- Subset out dates and time of interest

In [ ]:
def prep_station(stationid):
    """Add z0/zm, merge NLDAS ET and subset by date and time of day.

    Returns (df_combined, station_lat, station_lon).
    """
    veg_type, default_crop_height = VEG_PARAMS.get(stationid, [None, 1.5])
    if stationid not in VEG_PARAMS:
        print(f"WARNING: {stationid} not in VEG_PARAMS, using veg_type=None and crop height 1.5 m")

    df_temp, params, irga_height, station_lat, station_lon = read_station_data(stationid)

    df_all = df_temp.merge(params[['measurement_height', 'crop_height_m']],
                           left_index=True,
                           right_index=True,
                           how='left'
                           )

    if df_all['measurement_height'].isnull().all():
        if irga_height is None:
            raise ValueError(f"No measurement height or irgason height for {stationid}")
        df_all['measurement_height'] = irga_height
    if df_all['crop_height_m'].isnull().all():
        df_all['crop_height_m'] = default_crop_height

    # computer z0 and zm
    df_all = compute_aerodynamic_params(df_all,
                                        inst_height = 'measurement_height',
                                        crop_height='crop_height_m',
                                        veg_type=veg_type)

    # check this....
    check = df_all['zm']>df_all['z0']*12.5
    print(f'Percent of records outside roughness layer bounds {round(100*check.sum()/len(df_all), 1)}')

    # merge with nldas data
    nldas = pd.read_csv(NLDAS_DIR / f"{stationid}_nldas.csv",
                        parse_dates=['timestamp_local'],
                        index_col='timestamp_local')
    nldas['timestamp_end'] = pd.to_datetime(nldas['timestamp_end'])
    nldas.rename(columns={'eto_grass':'nldas_et'}, inplace=True)

    for col in nldas.columns:
        if "imestamp" in col:
            nldas[col] = pd.to_datetime(nldas[col], errors="coerce")
        else:
            nldas[col] = pd.to_numeric(nldas[col], errors="coerce")

    nldas_resample = nldas.resample(DATA_FREQ).interpolate(numeric_only=True)
    df_combined = pd.merge(df_all,nldas_resample['nldas_et'],
                        left_index=True,
                        right_index=True,
                        how='left')
    assert df_combined.nldas_et.isna().sum() == 0, 'Some records missing nldas ET data'

    # subset out by dates and time of day
    mask = (df_combined.index>=pd.to_datetime(START)) & (df_combined.index<=pd.to_datetime(END))
    df_combined = df_combined[mask]
    mask = (df_combined.index.hour>=START_TIME) & (df_combined.index.hour<=END_TIME)
    df_combined = df_combined[mask]
    df_combined = df_combined.sort_index()

    if df_combined.empty:
        raise ValueError(f"No records for {stationid} between {START} and {END}")

    return df_combined, station_lat, station_lon

## Month-by-month Footprint Calculation

`run_station` processes one station. For each month the workflow is:

1. **`build_climatology`** – runs the Kljun et al. 2-D FFP model on every
   valid half-hour and stacks results into an xarray Dataset.
2. **`summarize_periods`** – aggregates the per-timestep footprint stack into
   daily and monthly summary rasters (arithmetic mean and ET-flux-weighted).
3. **`export_rasters_geotiff`** – writes the ET-weighted rasters to
   `{stationid}/weighted` and the unweighted means to `{stationid}/non_weighted`,
   in the local UTM zone at the station's lat/lon origin.

`gc.collect()` after each month releases the large xarray objects immediately
rather than waiting for Python's garbage collector.

In [ ]:
REQ_FIELDS = ["mo_length_1_1_1",
              "v_sigma_1_1_1",
              "ustar_1_1_1",
              "wd_1_1_1",
              "ws_1_1_1",
              ]


def run_station(stationid):
    """Calculate and export weighted and unweighted footprints for one station.

    Returns (results, domain_results): per-month status dicts and a list of
    daily domain-coverage DataFrames.
    """
    weighted_out = FP_DIR / stationid / "weighted"
    non_weighted_out = FP_DIR / stationid / "non_weighted"
    weighted_out.mkdir(parents=True, exist_ok=True)
    non_weighted_out.mkdir(parents=True, exist_ok=True)

    df_combined, station_lat, station_lon = prep_station(stationid)

    months = pd.period_range(start=df_combined.index.min(), end=df_combined.index.max(), freq="M")
    print(f"Processing {len(months)} months: {months[0]} – {months[-1]}")

    results = []    # collect per-month summaries for the table below
    skipped = []
    domain_results = []

    for period in months:
        label       = str(period)               # e.g. "2023-01"
        yyyymm      = label.replace("-", "")    # e.g. "202301"

        df_mon = df_combined.loc[label].copy().sort_index()
        if df_mon.empty:
            print(f"[{label}] Skipped: No raw data found.")
            skipped.append(label)
            continue

        valid_mask_mon = df_mon[REQ_FIELDS].notna().all(axis=1)
        n_valid = int(valid_mask_mon.sum())

        print(f"  {label}: {len(df_mon):5d} records,  {n_valid:4d} with required fields … ", end="", flush=True)

        if n_valid < 10:
            print("SKIPPED (not enough records with required fields)")
            skipped.append(label)
            results.append({"stationid": stationid, "month": label, "n_req_fields": n_valid, "status": "skipped",
                            "n_daily_tifs": 0, "n_monthly_tifs": 0})
            continue

        clim = out = None
        try:
            # ── Step 1: build footprint climatology ──────────────────────────────
            clim = build_climatology(
                df_mon,
                model_type = 'ffp_xr', #ffp_xr, ffp (which is from improved_ffp), km, ls, wang
                ustar = 'ustar_1_1_1',
                ol = 'mo_length_1_1_1',
                umean = 'ws_1_1_1',
                sigmav = 'v_sigma_1_1_1',
                wind_dir = 'wd_1_1_1',
                zm = 'zm',
                z0 = 'z0',
                h = 2000.0,
                dx=DX,
                dy=DY,
                domain=DOMAIN,
                rslayer = False # whether to calculate footprint even if zm within roughness sublayer
            )

            # ── Step 2: summarize into daily + monthly rasters ───────────────────
            out = summarize_periods(
                clim,
                df_mon,
                et_source="nldas_et",
                is_le = False,  # if using le instead of et value
                calc_et_weighted = True,
                daily=True,
                monthly=True,
                normalize_each_time=True,
            )

            # ── Step 3: write GeoTIFFs (weighted and unweighted from the same run) ──
            export_kwargs = dict(
                station_lat=station_lat,
                station_lon=station_lon,
                crs_out="auto",    # auto-selects UTM zone from lat/lon
                prefix=f"ffp_{stationid}",
                dtype="float32",
                nodata=0.0,
            )
            export_rasters_geotiff(clim, out, out_dir=weighted_out,
                                   which=("daily_etw", "monthly_etw"), **export_kwargs)
            export_rasters_geotiff(clim, out, out_dir=non_weighted_out,
                                   which=("daily_mean", "monthly_mean"), **export_kwargs)

            w_tifs  = list(weighted_out.glob(f"ffp_{stationid}_*{yyyymm}*.tif"))
            nw_tifs = list(non_weighted_out.glob(f"ffp_{stationid}_*{yyyymm}*.tif"))
            n_daily_tifs   = len([f for f in w_tifs  if "daily"   in f.name])
            n_monthly_tifs = len([f for f in w_tifs  if "monthly" in f.name])
            n_daily_nw     = len([f for f in nw_tifs if "daily"   in f.name])
            n_monthly_nw   = len([f for f in nw_tifs if "monthly" in f.name])

            print(f"done  ({clim.ts_len} timesteps → weighted {n_daily_tifs}d/{n_monthly_tifs}m, "
                  f"unweighted {n_daily_nw}d/{n_monthly_nw}m TIFs)")
            results.append({"stationid": stationid, "month": label, "n_req_fields": n_valid, "status": "ok",
                            "ts_len": clim.ts_len,
                            "n_daily_tifs": n_daily_tifs, "n_monthly_tifs": n_monthly_tifs,
                            "n_daily_nw_tifs": n_daily_nw, "n_monthly_nw_tifs": n_monthly_nw})
            domain_results.append(out.daily_domain_coverage.assign(stationid=stationid))

        except Exception as exc:
            print(f"ERROR: {exc}")
            skipped.append(label)
            results.append({"stationid": stationid, "month": label, "n_req_fields": n_valid, "status": f"error: {exc}",
                            "n_daily_tifs": 0, "n_monthly_tifs": 0})

        finally:
            del clim, out
            gc.collect()

    print(f"{stationid} finished. Skipped months: {skipped if skipped else 'none'}")
    return results, domain_results

## Run All Stations

A station that fails (missing NLDAS file, no measurement height, no data in the
date window, etc.) is reported in `failed_stations` and the loop moves on.

In [ ]:
all_results = []
all_domain = []
failed_stations = {}

for stationid in STATIONS:
    print(f"\n===== {stationid} =====")
    try:
        results, domain_results = run_station(stationid)
        all_results += results
        all_domain += domain_results
    except Exception as exc:
        print(f"{stationid} FAILED: {exc}")
        failed_stations[stationid] = str(exc)
    finally:
        gc.collect()

results_df = pd.DataFrame(all_results)
final_domain_results = pd.concat(all_domain) if all_domain else pd.DataFrame()
print(f"\nFailed stations: {failed_stations if failed_stations else 'none'}")

## Review Results

- Goal from Volk is at least 5 hours per day and 260 hours per month
- This is equivalent to 10 half-hours/day and 520 hours/month

In [ ]:
# export daily and monthly domain results per station and review results
for stationid, dom in final_domain_results.groupby('stationid'):
    dom = dom.drop(columns='stationid')
    dom.to_csv(DOMAIN_DIR / f'{stationid}_daily_domain_results.csv')
    dom.resample('ME').sum().to_csv(DOMAIN_DIR / f'{stationid}_monthly_domain_results.csv')

print(results_df.to_string(index=False))

print(final_domain_results.groupby('stationid').agg(
    min_coverage_pct=('min_coverage_pct', 'min'),
    min_valid_timesteps=('valid_timesteps', 'min'),
).round(2))

temp = final_domain_results[final_domain_results.mean_coverage_pct.notna()]
groups = list(temp.groupby('stationid'))
plt.boxplot([g.mean_coverage_pct for _, g in groups], tick_labels=[sid for sid, _ in groups])
plt.ylabel('Mean daily coverage')
plt.show()

In [ ]:
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

for stationid, res_plot in final_domain_results.groupby('stationid'):
    if "valid_timesteps" not in res_plot.columns:
        continue
    res_plot = res_plot.copy()

    # 1. Extract datetime from index or 'time' column
    if "time" in res_plot.columns:
        dt_index = pd.to_datetime(res_plot["time"])
    else:
        dt_index = pd.to_datetime(res_plot.index)

    res_plot["year"] = dt_index.year
    res_plot["month_n"] = dt_index.month

    # 2. Aggregate daily counts to monthly sums using pivot_table
    pivot = res_plot.pivot_table(
        index="year",
        columns="month_n",
        values="valid_timesteps",
        aggfunc="sum",
    )

    # 3. Ensure all 12 months (1-12) exist in order, filling missing months with 0
    pivot = pivot.reindex(columns=range(1, 13), fill_value=0).fillna(0)

    # 4. Plot heatmap
    fig, ax = plt.subplots(figsize=(12, max(2, len(pivot) * 1.2)))
    im = ax.imshow(pivot.values, aspect="auto", cmap="YlGn", vmin=0)

    ax.set_xticks(range(12))
    ax.set_xticklabels(month_labels)
    ax.set_yticks(range(len(pivot)))
    ax.set_yticklabels(pivot.index)

    # 5. Annotate monthly values on grid
    for r, yr in enumerate(pivot.index):
        for c, mon in enumerate(pivot.columns):
            val = int(pivot.loc[yr, mon])
            if val > 0:  # Only annotate non-zero months
                ax.text(
                    c,
                    r,
                    str(val),
                    ha="center",
                    va="center",
                    fontsize=9,
                    color="black" if val > 200 else "gray",
                )

    plt.colorbar(im, ax=ax, label="Valid half-hours")
    ax.set_title(f"{stationid}: {DATA_FREQ} records valid for footprint calculation")
    fig.tight_layout()
    # save before show; show() clears the figure
    plt.savefig(DOMAIN_DIR / f"{stationid}_valid_timesteps_heatmap.png", dpi=300)
    plt.show()

## Review Wind Rose

In [ ]:
from windrose import WindroseAxes

WINDROSE_STATION = STATIONS[0]
target_date = pd.to_datetime('2025-07-13')

# records whose datetime_start falls on target_date
first_end = target_date + pd.Timedelta(DATA_FREQ)
wind_query = f"""SELECT datetime_end, wd_1_1_1, ws_1_1_1
                FROM groundwater.metflux_qc
                WHERE stationid = '{WINDROSE_STATION}'
                AND datetime_end >= '{first_end}'
                AND datetime_end < '{first_end + pd.Timedelta(days=1)}'
            """
hourly_subset = pd.read_sql(wind_query, engine)

fig = plt.figure(figsize=(8, 8), dpi=80)
ax = WindroseAxes.from_ax(fig=fig)
ax.bar(hourly_subset['wd_1_1_1'], hourly_subset['ws_1_1_1'], normed=True, opening=0.8, edgecolor='white')
ax.set_legend()
plt.title(f'{WINDROSE_STATION} {target_date.date()}')

windrose_path = DOMAIN_DIR / f"{WINDROSE_STATION}_{target_date.date()}_windrose.png"
plt.savefig(windrose_path, dpi=300)
plt.show()
print(f"Windrose plot saved to {windrose_path}")